# MammalianLifespan

## Index
1. [Instantiate model class](#Instantiate-model-class)
2. [Define clock metadata](#Define-clock-metadata)
3. [Download clock dependencies](#Download-clock-dependencies)
5. [Load features](#Load-features)
6. [Load weights into base model](#Load-weights-into-base-model)
7. [Load reference values](#Load-reference-values)
8. [Load preprocess and postprocess objects](#Load-preprocess-and-postprocess-objects)
10. [Check all clock parameters](#Check-all-clock-parameters)
10. [Basic test](#Basic-test)
11. [Save torch model](#Save-torch-model)
12. [Clear directory](#Clear-directory)

Let's first import some packages:

In [1]:
import os
import inspect
import shutil
import json
import math
import torch
import pandas as pd
import pyaging as pya

## Instantiate model class

In [2]:
def print_entire_class(cls):
    source = inspect.getsource(cls)
    print(source)

print_entire_class(pya.models.MammalianLifespan)

class MammalianLifespan(pyagingModel):
    def __init__(self):
        super().__init__()

    def preprocess(self, x):
        return x

    def postprocess(self, x):
        """
        Applies an anti-log transformation.
        """
        return torch.exp(x)



In [3]:
model = pya.models.MammalianLifespan()

## Define clock metadata

In [4]:
model.metadata["clock_name"] = "mammalianlifespan"
model.metadata["data_type"] = "DNA methylation"  # Paper: The model is based on DNA methylation measurements.
model.metadata["species"] = "multiple species"  # Paper: The study samples are multi.
model.metadata["year"] = 2024
model.metadata["approved_by_author"] = "⌛"
model.metadata["citation"] = "Li, Caesar Z., et al. \"Epigenetic predictors of species maximum life span and other life-history traits in mammals.\" Science Advances 10.23 (2024): eadm7273."
model.metadata["doi"] = "https://doi.org/10.1126/sciadv.adm7273"
model.metadata["notes"] = "Pan-mammalian tissue-agnostic elastic-net predictor fitted to log species maximum life span from conserved CpG methylation; pyaging exponentiates the linear output to years."
model.metadata["research_only"] = None
model.metadata["tissue"] = ["multi-tissue"]  # Paper: These samples spanned 59 unique tissue types and originated from 348 distinct mammalian species across 25 taxonomic orders.
model.metadata["predicts"] = ["species maximum lifespan"]  # Paper: We will refer to the predicted maximum life span, expressed in log years, as either the epigenetic maximum life span or DNAm maximum life span.
model.metadata["training_target"] = ["species maximum lifespan"]  # Paper: We used three distinct penalized regression models to predict the log-transformed values of maximum life span, gestation time, and age at sexual maturity for each species.
model.metadata["unit"] = ["years"]  # Paper: We will refer to the predicted maximum life span, expressed in log years, as either the epigenetic maximum life span or DNAm maximum life span.
model.metadata["model_type"] = "elastic net regression"  # Paper: First, we used elastic net regression models to predict maximum life span using both CpG methylation data and taxonomic order indicators.
model.metadata["platform"] = ["Horvath MammalMethylChip40"]  # Paper: All data were generated using the mammalian methylation array (HorvathMammalMethylChip40), which provides high sequencing depth of highly conserved CpGs in mammals.
model.metadata["population"] = "multiple mammalian species"  # Paper: Leveraging our publicly accessible data from the Mammalian Methylation Consortium, we focused on highly conserved cytosine methylation profiles from n = 15,000 DNA samples. These samples spanned 59 unique tissue types and originated from 348 distinct mammalian species across 25 taxonomic orders.
model.metadata["journal"] = "Science Advances"
model.metadata["last_author"] = "Steve Horvath"
model.metadata["n_features"] = 152  # Paper: The official LifespanPredictor_40K_Li2021.csv contains 152 nonzero non-intercept CpG coefficients.
model.metadata["citations"] = 5
model.metadata["citations_date"] = "2026-07-05"


## Download clock dependencies

#### Download GitHub repository

In [5]:
github_url = "https://github.com/caeseriousli/MammalianMethylationPredictors.git"
github_folder_name = github_url.split('/')[-1].split('.')[0]
os.system(f"git clone {github_url}")

0

## Load features

#### From CSV file

In [6]:
df = pd.read_csv('MammalianMethylationPredictors/Predictors/LifespanPredictor_40K_Li2021.csv')
df['feature'] = df['CpG']
df['coefficient'] = df['Coefficient']
df = df[df.Coefficient != 0]

model.features = df['feature'][1:].tolist()

## Load weights into base model

In [7]:
weights = torch.tensor(df['coefficient'][1:].tolist()).unsqueeze(0)
intercept = torch.tensor([df['coefficient'][0]])

#### Linear model

In [8]:
base_model = pya.models.LinearModel(input_dim=len(model.features))

base_model.linear.weight.data = weights.float()
base_model.linear.bias.data = intercept.float()

model.base_model = base_model

## Load reference values

In [9]:
model.reference_values = [0.5] * len(model.features)

## Load preprocess and postprocess objects

In [10]:
model.preprocess_name = None
model.preprocess_dependencies = None

In [11]:
model.postprocess_name = 'anti_log'
model.postprocess_dependencies = None

## Check all clock parameters

In [12]:
pya.utils.print_model_details(model)


%==================================== Model Details ====================================%
Model Attributes:

training: True
metadata: {'approved_by_author': '⌛',
 'citation': 'Li, Caesar Z., et al. "Epigenetic predictors of species maximum '
             'lifespan and other life history traits in mammals." bioRxiv '
             '(2023): 2023-11.',
 'clock_name': 'mammalianlifespan',
 'data_type': 'methylation',
 'doi': 'https://doi.org/10.1101/2023.11.02.565286',
 'notes': None,
 'research_only': None,
 'species': 'multi',
 'version': None,
 'year': 2023}
reference_values: [0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5]... [Total elements: 152]
preprocess_name: None
preprocess_dependencies: None
postprocess_name: 'anti_log'
postprocess_dependencies: None
features: ['cg00039845', 'cg00300233', 'cg00810217', 'cg01020408', 'cg01266508', 'cg01309159', 'cg01786675', 'cg02476543', 'cg0257

## Normal feature ranges

In [ ]:
# Units and plausibility ranges come from the package registry, keyed by feature name.
feature_ranges = pya.utils.resolve_feature_ranges(model.features, model.metadata["data_type"])
model.feature_units = [record["unit"] for record in feature_ranges]
pd.DataFrame.from_records(feature_ranges).head()

## Basic test

In [ ]:
# Exercise the clock with values in the middle of each feature's expected range.
records = pya.utils.resolve_feature_ranges(model.features, model.metadata["data_type"])
midpoints = [
    (record["low"] + record["high"]) / 2 if math.isfinite(record["high"]) else max(record["low"], 1.0)
    for record in records
]
input = torch.tensor([midpoints] * 10, dtype=torch.float64)
model.eval()
model.to(torch.float64)
pred = model(input)
pred

## Save torch model

In [14]:
torch.save(model, f"../weights/{model.metadata['clock_name']}.pt")

## Clear directory
<a id="10"></a>

In [15]:
# Function to remove a folder and all its contents
def remove_folder(path):
    try:
        shutil.rmtree(path)
        print(f"Deleted folder: {path}")
    except Exception as e:
        print(f"Error deleting folder {path}: {e}")

# Get a list of all files and folders in the current directory
all_items = os.listdir('.')

# Loop through the items
for item in all_items:
    # Check if it's a file and does not end with .ipynb
    if os.path.isfile(item) and not item.endswith('.ipynb'):
        os.remove(item)
        print(f"Deleted file: {item}")
    # Check if it's a folder
    elif os.path.isdir(item):
        remove_folder(item)

Deleted folder: MammalianMethylationPredictors
